# Notebook 04 — Multi-Hop Retrieval & Agent Pipeline

**Goal**: Build a RAG agent that verifies Romanian-language claims via:
1. **Decompose** — LLM breaks the claim into 2–4 factual sub-questions.
2. **Retrieve** — Hybrid BM25 + dense search (multilingual-e5-base) answers each sub-question.
3. **Synthesize** — Snippets are aggregated into an evidence history.
4. **Verify** — CoT reasoning assigns a final veracity label + Romanian justification.

**Retrieval**: `intfloat/multilingual-e5-base` embeddings + BM25, fused via Reciprocal Rank Fusion.

In [ ]:
# %pip install sentence-transformers rank-bm25 faiss-cpu transformers accelerate pandas tqdm

In [ ]:
import os
import sys
import json
from pathlib import Path
from typing import List, Tuple

import pandas as pd
import torch
from tqdm.auto import tqdm

sys.path.insert(0, str(Path('..').resolve()))
from src.tools import (
    HybridRetriever,
    decompose_claim,
    aggregate_evidence,
    build_verify_prompt,
    parse_verdict,
)
from src.metrics import classification_metrics, print_classification_metrics, retrieval_metrics

PROCESSED_DIR = Path('../data/processed')
MODELS_DIR    = Path('../data/models')

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')

## 1. Load Data

In [ ]:
train_df = pd.read_csv(PROCESSED_DIR / 'train.csv')
val_df   = pd.read_csv(PROCESSED_DIR / 'val.csv')
test_df  = pd.read_csv(PROCESSED_DIR / 'test.csv')

# Build evidence corpus from train + val splits.
# Using only 'train' would miss evidence from topics that only appear in val;
# val labels are never exposed — only the evidence_text field is indexed.
corpus = (
    pd.concat([train_df, val_df])['evidence_text']
    .dropna()
    .str.strip()
    .loc[lambda s: s.str.len() > 20]
    .unique()
    .tolist()
)
print(f'Evidence corpus size: {len(corpus)}')
print(f'Test set size: {len(test_df)}')

## 2. Build Hybrid Retriever

In [ ]:
retriever = HybridRetriever(corpus=corpus, device=DEVICE)
print('Hybrid retriever ready (BM25 + multilingual-e5-base).')

## 3. Load LLM for Decomposition and Verification

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel

LLM_MODEL = 'meta-llama/Llama-3.1-8B-Instruct'
LORA_PATH = str(MODELS_DIR / 'llama_lora_verdict')

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

print(f'Loading {LLM_MODEL} …')
tokenizer = AutoTokenizer.from_pretrained(LLM_MODEL)
tokenizer.pad_token = tokenizer.eos_token

base_llm = AutoModelForCausalLM.from_pretrained(
    LLM_MODEL,
    quantization_config=bnb_config,
    device_map='auto',
    trust_remote_code=True,
)

# Load LoRA weights if available
if Path(LORA_PATH).exists():
    llm = PeftModel.from_pretrained(base_llm, LORA_PATH)
    print('LoRA adapters loaded.')
else:
    llm = base_llm
    print('Using base model (no LoRA found).')

llm.eval()

## 4. LLM Callable Wrapper

In [ ]:
def llm_generate(prompt: str, max_new_tokens: int = 512) -> str:
    """Simple callable used by `decompose_claim` and verification."""
    inputs = tokenizer(
        prompt,
        return_tensors='pt',
        truncation=True,
        max_length=2048,
    )
    inputs = {k: v.to(llm.device) for k, v in inputs.items()}
    with torch.no_grad():
        output_ids = llm.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.1,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
        )
    new_tokens = output_ids[0][inputs['input_ids'].shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True)

## 5. Multi-Hop Agent — Single Claim Pipeline

In [ ]:
def multihop_verify(claim: str, top_k_per_hop: int = 3, n_subquestions: int = 3) -> dict:
    """
    Full multi-hop pipeline for a single claim.
    Returns a dict with sub_questions, all_snippets, evidence_str, label, justification.
    """
    # --- Step 1: Decompose ---
    sub_questions = decompose_claim(claim, llm_fn=llm_generate, n=n_subquestions)

    # --- Step 2: Retrieve for each sub-question ---
    all_snippets: List[Tuple[str, float]] = []
    seen_texts = set()
    for q in sub_questions:
        results = retriever.retrieve(q, top_k=top_k_per_hop)
        for text, score in results:
            if text not in seen_texts:
                all_snippets.append((text, score))
                seen_texts.add(text)

    # Also retrieve directly on the original claim
    for text, score in retriever.retrieve(claim, top_k=top_k_per_hop):
        if text not in seen_texts:
            all_snippets.append((text, score))
            seen_texts.add(text)

    # Sort by score descending
    all_snippets.sort(key=lambda x: x[1], reverse=True)

    # --- Step 3: Aggregate evidence ---
    evidence_str = aggregate_evidence(all_snippets, max_tokens=1500)

    # --- Step 4: CoT Verification ---
    verify_prompt = build_verify_prompt(claim, evidence_str)
    raw_output = llm_generate(verify_prompt, max_new_tokens=400)
    label, justification = parse_verdict(raw_output)

    return {
        'claim': claim,
        'sub_questions': sub_questions,
        'retrieved_snippets': [t for t, _ in all_snippets],
        'evidence_str': evidence_str,
        'raw_output': raw_output,
        'pred_label': label,
        'justification': justification,
    }


# Quick smoke-test on one example
if len(test_df) > 0:
    sample_row = test_df.iloc[0]
    demo = multihop_verify(sample_row['claim_text'])
    print('Sub-questions:', demo['sub_questions'])
    print('Predicted label:', demo['pred_label'])
    print('Justification:', demo['justification'][:300])

## 6. Run Pipeline on Full Test Set

In [ ]:
agent_records = []

for _, row in tqdm(test_df.iterrows(), total=len(test_df), desc='Multi-hop agent'):
    result = multihop_verify(str(row['claim_text']))
    result['true_label']    = row['veracity_label']
    result['gold_evidence'] = str(row.get('evidence_text', ''))
    agent_records.append(result)

agent_df = pd.DataFrame(agent_records)
agent_df.to_json(
    PROCESSED_DIR / 'agent_results.jsonl',
    orient='records',
    lines=True,
    force_ascii=False,
)
print(f'Agent results saved: {len(agent_df)} rows')

## 7. Classification Metrics

In [ ]:
y_true = agent_df['true_label'].tolist()
y_pred = agent_df['pred_label'].tolist()

agent_class_metrics = classification_metrics(y_true, y_pred)
print('=== Multi-Hop Agent Classification Metrics ===')
print_classification_metrics(agent_class_metrics)

## 8. Retrieval Metrics

In [ ]:
retrieved_snippets_all = agent_df['retrieved_snippets'].tolist()
gold_evidence_all      = agent_df['gold_evidence'].tolist()

ret_metrics = retrieval_metrics(
    retrieved_snippets=retrieved_snippets_all,
    reference_evidence=gold_evidence_all,
)
print('Retrieval metrics:', ret_metrics)

## 9. Save All Agent Metrics

In [ ]:
all_agent_metrics = {
    'classification': agent_class_metrics,
    'retrieval': ret_metrics,
}

with open(PROCESSED_DIR / 'agent_metrics.json', 'w', encoding='utf-8') as f:
    json.dump(all_agent_metrics, f, indent=2, ensure_ascii=False)

print('Agent metrics saved to data/processed/agent_metrics.json')